# Logger levels

Regression check for [#300](https://github.com/sogno-platform/dpsim/issues/300):
the log level handed to a component or simulation constructor must be the level its
log file is written at. Lines below `SPDLOG_ACTIVE_LEVEL` are removed at compile
time, so `debug` lines are only expected from a build that compiled them in
(`Debug` builds, or `-DDPSIM_LOG_LEVEL=DEBUG`); every other check holds for every
build type.

In [ ]:
import os
import tempfile

import dpsimpy

log_dir = tempfile.mkdtemp(prefix="dpsim-logger-level-")
dpsimpy.Logger.set_log_dir(log_dir)

compiled = dpsimpy.Logger.compiled_level()
debug_compiled = int(compiled) <= int(dpsimpy.LogLevel.debug)
print("compiled level:", compiled, "- debug lines expected:", debug_compiled)

In [ ]:
def log_lines(name, marker):
    path = os.path.join(log_dir, name + ".log")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return [line for line in f if line.startswith(marker)]


def run(tag, level):
    """Simulate a source and a resistor, all loggers at the same level."""
    gnd = dpsimpy.dp.SimNode.gnd
    n1 = dpsimpy.dp.SimNode(tag + "_n1")
    vs = dpsimpy.dp.ph1.VoltageSource(tag + "_vs", level)
    vs.set_parameters(V_ref=complex(10, 0))
    r1 = dpsimpy.dp.ph1.Resistor(tag + "_r1", level)
    r1.set_parameters(R=1)
    vs.connect([gnd, n1])
    r1.connect([n1, gnd])

    system = dpsimpy.SystemTopology(50, [gnd, n1], [vs, r1])
    sim = dpsimpy.Simulation(tag, level)
    sim.set_system(system)
    sim.set_time_step(1e-3)
    sim.set_final_time(5e-3)
    sim.run()
    return tag + "_r1", tag + "_vs", tag

In [ ]:
r1, vs, sim = run("LoggerLevelDebug", dpsimpy.LogLevel.debug)

assert log_lines(r1, "[I]"), "info lines missing from the component log"
assert log_lines(sim, "[I]"), "info lines missing from the simulation log"
if debug_compiled:
    assert log_lines(r1, "[D]"), "debug requested but no debug lines written"
    assert log_lines(vs, "[D]"), "debug requested but no debug lines written"
    print("debug level honoured: ok")
else:
    assert not log_lines(r1, "[D]") and not log_lines(vs, "[D]")
    print("debug calls compiled out, none written: ok")

In [ ]:
r1, vs, sim = run("LoggerLevelInfo", dpsimpy.LogLevel.info)

assert log_lines(r1, "[I]") and log_lines(sim, "[I]")
assert not log_lines(r1, "[D]") and not log_lines(vs, "[D]")
print("info level writes info and nothing below it: ok")

In [ ]:
r1, vs, sim = run("LoggerLevelWarn", dpsimpy.LogLevel.warn)

for name in (r1, vs, sim):
    assert not log_lines(name, "[I]"), name + " wrote info lines at warn level"
    assert not log_lines(name, "[D]"), name + " wrote debug lines at warn level"
print("warn level stays quiet: ok")

In [ ]:
r1, vs, sim = run("LoggerLevelOff", dpsimpy.LogLevel.off)

assert log_lines(r1, "") is None and log_lines(vs, "") is None
print("off level opens no component log file: ok")